# RAG 다차원 평가

- 구분: 수강생 실습본
- 원본: [`05_rag_evaluation_extra.py`](05_rag_evaluation_extra.py)
- 연결 교안: [4-2-langfuse-observability-and-evaluation.pdf](../../99_pdf/Chapter02/day02/4-2-langfuse-observability-and-evaluation.pdf)

원본 Python 코드는 수정 없이 아래의 한 코드 셀에 담았습니다. 설명과 실행 준비는 별도 셀입니다. **모두 실행** 대신 필요한 셀을 직접 선택하세요.


## 학습 목표

1. 검색 컨텍스트에 따라 정확성·관련성·완성도가 달라지는 이유를 설명한다.
2. 코드를 실행하기 전 입력, 출력, 외부 호출과 파일 변경 가능성을 설명한다.
3. CodeMate가 만든 결과를 원본과 교안에 비춰 검토한다.


## 코드를 읽는 순서

가상 문서 → retrieve → generate → judge → 정상·빈·오염 컨텍스트 순서로 읽는다.

교안은 개념과 목표를 설명합니다. 코드에서 실제 구현된 부분과 TODO를 구분하세요.


## 실행 준비

저장소 안에서 VS Code로 열고 프로젝트 `.venv` 커널을 선택합니다. 의존성은 `uv sync`로 준비합니다. 사내망에서는 사내 LLM 환경을, 사외에서는 허용된 provider를 준비합니다. 자격증명 값은 셀에 적지 않습니다. 아래 준비 셀은 저장소 루트를 찾고 `__file__`을 원본 Python 파일 경로로 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys
_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents) if (p / "pyproject.toml").is_file() and (p / "config" / "llm.py").is_file()), None)
if _repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 열어 주세요.")
os.chdir(_repo)
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
__file__ = str(_repo / 'Chapter02/05-observability/05_rag_evaluation_extra.py')
print(f"실행 기준 폴더: {_repo}")


## 원본 코드 — 직접 실행

**조건과 부작용:** 생성과 평가에 여러 LLM 호출이 쓰이고 Trace·Score가 전송된다. 문서는 모의 자료이며 점수는 매번 달라질 수 있다.

준비 셀을 먼저 실행하고 필요한 서비스·설정을 확인하세요. 다음 셀은 `manual-run`으로 표시했습니다. 원본 전체가 한 번에 실행되므로 API 요청, 파일 변경 또는 대기가 생길 수 있습니다.


In [ ]:
"""
RAG 파이프라인 다차원 평가 (Extra)

구조:
  1. @observe로 retrieve → generate 단계를 각각 span으로 기록
  2. LLM Judge가 정확성·관련성·완성도 3개 차원을 독립적으로 채점
  3. 의도적으로 오염된 컨텍스트를 섞어 점수가 실제로 달라지는 것을 확인

점수가 달라지는 이유:
  - CASE A: 정확한 컨텍스트 → 고득점
  - CASE B: 빈 컨텍스트 (검색 실패 시뮬레이션) → 낮은 관련성
  - CASE C: 오염된 컨텍스트 (엉뚱한 문서) → 낮은 정확성

실행:
  python Chapter02/05-observability/05_rag_evaluation_extra.py
"""

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client, observe
from langfuse.langchain import CallbackHandler
from pydantic import BaseModel, Field

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()


# ── 가상 문서 저장소 ────────────────────────────────────────────────────────────
DOCS = {
    'langgraph': (
        'LangGraph는 LangChain 위에서 동작하는 상태 기반 멀티에이전트 프레임워크다. '
        'StateGraph로 노드(함수)와 엣지(전환 조건)를 정의하며, MemorySaver로 체크포인트를 저장한다. '
        'LangChain 0.x의 AgentExecutor를 대체하며 복잡한 루프·분기 흐름을 지원한다.'
    ),
    'rag': (
        'RAG(Retrieval-Augmented Generation)는 외부 문서를 검색해 LLM의 답변 근거로 활용하는 기법이다. '
        '벡터 임베딩으로 유사 문서를 찾고, 해당 내용을 프롬프트 컨텍스트에 삽입한다. '
        '환각(hallucination)을 줄이고 최신 정보를 반영할 수 있다는 장점이 있다.'
    ),
    'noise': (
        '오늘 서울 날씨는 맑고 기온은 23도이다. '
        '내일은 구름이 많고 오후에 비가 올 수 있다. '
        '주말에는 기온이 낮아질 전망이다.'
    ),
}


# ── 평가 결과를 구조화하기 위한 Pydantic 모델 ──────────────────────────────────
class EvalScore(BaseModel):
    correctness: float = Field(description='사실 정확성 0.0~1.0')
    relevance: float = Field(description='질문과의 관련성 0.0~1.0')
    completeness: float = Field(description='답변 완성도 0.0~1.0')
    reason: str = Field(description='채점 근거 한 줄')


# ── RAG 단계별 함수 ─────────────────────────────────────────────────────────────

@observe(name='retrieve')
def retrieve(query: str, doc_key: str) -> str:
    """doc_key로 문서를 가져온다. 실제 환경에서는 벡터 검색으로 대체."""
    return DOCS.get(doc_key, '')


@observe(name='generate')
def generate(question: str, context: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages([
        ('system', '아래 컨텍스트만 참고해 질문에 답하세요. 컨텍스트가 없으면 모른다고 하세요.\n\n{context}'),
        ('human', '{question}'),
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke(
        {'context': context, 'question': question},
        config={'callbacks': [handler]},
    )


@observe(name='rag-pipeline')
def rag_pipeline(question: str, doc_key: str, handler: CallbackHandler) -> tuple[str, str]:
    context = retrieve(question, doc_key)
    answer = generate(question, context, handler)
    return answer, context


# ── LLM Judge ──────────────────────────────────────────────────────────────────

def judge(question: str, context: str, answer: str) -> EvalScore:
    # structured output: LLM이 EvalScore 형식으로만 응답하도록 강제
    judge_llm = llm.with_structured_output(EvalScore)
    prompt = ChatPromptTemplate.from_messages([
        ('system',
         '질문, 참고 컨텍스트, 답변을 보고 3개 항목을 0.0~1.0으로 채점하세요.\n'
         '- correctness: 컨텍스트 기준으로 답변이 사실적으로 맞는지\n'
         '- relevance: 답변이 질문에 직접 답하는지\n'
         '- completeness: 컨텍스트의 핵심 정보를 빠짐없이 담았는지'),
        ('human', '질문: {question}\n\n컨텍스트: {context}\n\n답변: {answer}'),
    ])
    chain = prompt | judge_llm
    return chain.invoke({'question': question, 'context': context, 'answer': answer})


# ── 테스트 케이스 ───────────────────────────────────────────────────────────────

test_cases = [
    {
        'name': 'CASE A — 정확한 컨텍스트',
        'question': 'LangGraph가 뭔가요?',
        'doc_key': 'langgraph',   # 정확한 문서
    },
    {
        'name': 'CASE B — 빈 컨텍스트 (검색 실패)',
        'question': 'RAG의 장점은 무엇인가요?',
        'doc_key': 'missing',     # 존재하지 않는 키 → 빈 컨텍스트
    },
    {
        'name': 'CASE C — 오염된 컨텍스트 (엉뚱한 문서)',
        'question': 'RAG가 환각을 줄이는 원리를 설명해줘',
        'doc_key': 'noise',       # 날씨 문서 → 완전히 엉뚱함
    },
]


print('=== RAG 파이프라인 다차원 평가 ===\n')

for case in test_cases:
    print(f"[{case['name']}]")
    print(f"Q: {case['question']}")

    handler = CallbackHandler()
    answer, context = rag_pipeline(case['question'], case['doc_key'], handler)
    trace_id = handler.last_trace_id

    print(f"A: {answer[:120]}{'...' if len(answer) > 120 else ''}")

    # 3개 차원 채점
    score = judge(case['question'], context, answer)
    print(f"채점 → 정확성: {score.correctness:.1f} | 관련성: {score.relevance:.1f} | 완성도: {score.completeness:.1f}")
    print(f"근거: {score.reason}")

    # Langfuse에 차원별 점수 기록
    for name, value in [
        ('correctness', score.correctness),
        ('relevance', score.relevance),
        ('completeness', score.completeness),
    ]:
        langfuse.create_score(trace_id=trace_id, name=name, value=value)

    print()

langfuse.flush()
print('✓ 대시보드 [Scores] 탭에서 케이스별 차이를 확인하세요.')


## CodeMate에게 요청하고 검토하기

> 세 케이스의 예상 점수와 근거를 먼저 적고 실제 Judge 결과와 비교할 체크리스트를 만들어 줘.

CodeMate가 제안한 수정의 근거를 읽고, 실행 가능한 환경에서 출력·Trace·생성 파일을 확인하세요.


## 확인 질문

1. 빈 문서에서도 그럴듯한 답변이 가능한가? 오염 문서는 어느 점수에 영향을 주는가?
2. 실행 전 어떤 환경과 자료를 확인해야 하는가?
3. 결과가 기대와 다르면 어느 단계의 기록부터 볼 것인가?
